In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import networkx as nx
import random

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils.config import load_config
from src.utils.graph import radius_graph_pbc_batch
from src.pbc_config import min_image, wrap, BOX
from src.validation import get_tpcf
from src.dataset import create_dataloader

/home/bartb/venvs/boids/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cpu")

In [8]:
train_dl, val_dl, _ = create_dataloader(
    "../../Data",
    "../../Data",
    False,
    train_kwargs={"batch_size": 10},
    valid_kwargs={"batch_size": 10},
)

In [10]:
sample = next(iter(train_dl))

In [16]:
x1 = sample.x
x0 = torch.rand(50000, 3)
batch = torch.arange(10, dtype=int).repeat_interleave(5000)
t = torch.rand(10, 1)
t = t[batch]
x_t = min_image(x1-x0, **BOX) * t + x0
x_t = x_t.view(10, 5000, 3)

for i in range(x_t.shape[0]):
    graph = x_t[i]
    edge_index = radius_graph_pbc_batch(graph, 0.3, torch.zeros(5000, dtype=int), device)

    G = nx.Graph()
    G.add_edges_from(edge_index.T.tolist())
    sample_nodes = random.sample(list(G.nodes()), 50)
    approx_diameter = max(nx.eccentricity(G, v=sample_nodes).values())
    print(f"Approximate diameter: {approx_diameter}")

Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
Approximate diameter: 3
